# Solar Filament Segmentation - Version 20 offline fixed

This notebook does not install packages from the internet. It uses the native MediSegUNet architecture, PyTorch, OpenCV, NumPy and Pandas already available in the Kaggle image. Empty masks are skipped, compressed COCO RLE is generated without pycocotools, and the final CSV is validated before upload.

In [ ]:
import glob
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SIZE = (512, 512)
OUTPUT_SIZE = (2048, 2048)
THRESHOLD = 0.50
BATCH_SIZE = 8
NUM_WORKERS = 2
OUTPUT_PATH = '/kaggle/working/submission_v20_offline_fixed.csv'

def first_existing_directory(patterns):
    for pattern in patterns:
        matches = sorted(glob.glob(pattern, recursive=True))
        if matches:
            return matches[0]
    return None

TEST_IMG_DIR = first_existing_directory([
    '/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026/test/test_images',
    '/kaggle/input/**/test/test_images',
])
if TEST_IMG_DIR is None:
    raise FileNotFoundError('Could not find a test/test_images directory under /kaggle/input.')

CHECKPOINTS = sorted(
    glob.glob('/kaggle/input/**/best_solar_unet.pth', recursive=True)
    + glob.glob('/kaggle/input/**/best_solar_unet.pt', recursive=True)
    + glob.glob('/kaggle/input/**/best_solar_unet.zip', recursive=True)
)
if not CHECKPOINTS:
    raise FileNotFoundError('Attach best_solar_unet.pth or best_solar_unet.zip as a Kaggle input dataset.')

print({'device': str(DEVICE), 'test_dir': TEST_IMG_DIR, 'checkpoints': CHECKPOINTS})

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        mid_channels = mid_channels or out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=False):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True) if bilinear else nn.ConvTranspose2d(in_channels, in_channels // 2, 2, 2)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2) if bilinear else DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y, diff_x = x2.size(2) - x1.size(2), x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        return self.conv(torch.cat([x2, x1], dim=1))

class MediSegUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, bilinear=False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1, self.down2 = Down(64, 128), Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, 1)
    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3); x5 = self.down4(x4)
        return self.outc(self.up4(self.up3(self.up2(self.up1(x5, x4), x3), x2), x1))

def load_model(path):
    model = MediSegUNet().to(DEVICE)
    payload = torch.load(path, map_location='cpu')
    if isinstance(payload, dict) and 'state_dict' in payload: payload = payload['state_dict']
    if isinstance(payload, dict) and 'model_state_dict' in payload: payload = payload['model_state_dict']
    state = {k.removeprefix('module.').removeprefix('model.'): v for k, v in payload.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected: raise RuntimeError(f'Checkpoint mismatch: missing={missing[:3]}, unexpected={unexpected[:3]}')
    return model.eval()

models = [load_model(path) for path in CHECKPOINTS]
print(f'Loaded {len(models)} MediSegUNet checkpoint(s).')

In [ ]:
class TestDataset(Dataset):
    def __init__(self, image_dir):
        exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
        self.paths = sorted(p for ext in exts for p in glob.glob(os.path.join(image_dir, ext)))
        if not self.paths: raise FileNotFoundError(f'No images found in {image_dir}')
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        path = self.paths[index]; image = cv2.imread(path, cv2.IMREAD_COLOR)
        if image is None: raise ValueError(f'Could not read {path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, INPUT_SIZE, interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
        image = (image - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
        return torch.from_numpy(image.transpose(2, 0, 1)).float(), os.path.basename(path)

@torch.inference_mode()
def predict(loader):
    names, masks = [], []
    for images, batch_names in tqdm(loader, desc='Inference'):
        images = images.to(DEVICE, non_blocking=True)
        probs = torch.stack([torch.sigmoid(model(images)) for model in models]).mean(0)[:, 0].cpu().numpy()
        for name, prob in zip(batch_names, probs):
            prob = cv2.resize(prob, OUTPUT_SIZE, interpolation=cv2.INTER_LINEAR)
            names.append(name); masks.append((prob >= THRESHOLD).astype(np.uint8))
    return names, masks

loader = DataLoader(TestDataset(TEST_IMG_DIR), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
image_names, predicted_masks = predict(loader)
print(f'Predicted {len(predicted_masks)} image masks.')

In [ ]:
def coco_compress_counts(counts):
    chars = []
    for index, value in enumerate(counts):
        value = int(value)
        if index > 1: value -= int(counts[index - 2])
        value = ((-value) << 1 | 1) if value < 0 else (value << 1)
        while value >= 0x20:
            chars.append(chr((value & 0x1F) | 0x20 | 48)); value >>= 5
        chars.append(chr(value + 48))
    return ''.join(chars)

def encode_coco_rle(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    if mask.shape != OUTPUT_SIZE: raise ValueError(f'Expected {OUTPUT_SIZE}, got {mask.shape}')
    if not np.any(mask): return None
    pixels = np.asfortranarray(mask).reshape(-1, order='F')
    changes = np.flatnonzero(np.diff(np.concatenate(([0], pixels, [0]))))
    starts, ends = changes[0::2], changes[1::2]
    counts = []
    previous = 0
    for start, end in zip(starts, ends): counts.extend([int(start - previous), int(end - start)]); previous = int(end)
    if previous < pixels.size: counts.append(int(pixels.size - previous))
    return coco_compress_counts(counts)

def build_submission(names, masks, output_path):
    rows = []
    for name, mask in zip(names, masks):
        rle = encode_coco_rle(mask)
        if rle is None or not rle.strip(): continue
        rows.append({'filament_id': f'{Path(name).stem}_1', 'segmentation_rle': rle})
    frame = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
    if frame.empty: raise ValueError('No non-empty predictions were generated.')
    if frame.isna().any().any(): raise ValueError('Submission contains null values.')
    if frame['filament_id'].duplicated().any(): raise ValueError('Duplicate filament_id values.')
    if frame['segmentation_rle'].astype(str).str.strip().eq('').any(): raise ValueError('Empty segmentation_rle value.')
    frame.to_csv(output_path, index=False, lineterminator='\n')
    return frame

submission = build_submission(image_names, predicted_masks, OUTPUT_PATH)
print(f'Wrote {len(submission)} rows to {OUTPUT_PATH}')

In [ ]:
def validate_submission(path):
    frame = pd.read_csv(path, keep_default_na=False)
    expected = ['filament_id', 'segmentation_rle']
    if frame.columns.tolist() != expected: raise ValueError(f'Expected {expected}, got {frame.columns.tolist()}')
    if frame.empty or frame.isna().any().any(): raise ValueError('Empty or null submission after CSV round-trip.')
    if frame['filament_id'].astype(str).str.strip().eq('').any(): raise ValueError('Empty filament_id.')
    if frame['segmentation_rle'].astype(str).str.strip().eq('').any(): raise ValueError('Empty segmentation_rle.')
    if frame['filament_id'].duplicated().any(): raise ValueError('Duplicate filament_id.')
    return frame

validated = validate_submission(OUTPUT_PATH)
print('VALIDATED:', OUTPUT_PATH)
print(validated.head())